# Forecast plot & series diagnostic

This notebook runs:
1. **Forecast plot** – Load a trained model and plot last 28-day predicted vs actuals.
2. **Diagnose series** – Plot raw sales and print stats + suggestions for a (problem) series.

## Setup

**Run the three code cells below in order.** The first finds the project root and checks that `data/m5/sales_train_validation.csv` exists. The second imports; the third sets `paths`, `cfg`, and series IDs.

In [ ]:
import os
import sys
from pathlib import Path

# 1) Find project root (folder that contains both "src" and "data")
cwd = Path.cwd()
project_root = cwd if (cwd / "src").is_dir() else cwd.parent
if not (project_root / "src").is_dir():
    project_root = cwd.parent.parent  # e.g. notebook in notebooks/ and cwd was project root
sys.path.insert(0, str(project_root))

# 2) Validate: data must be at project_root/data/m5/
data_file = project_root / "data" / "m5" / "sales_train_validation.csv"
if not data_file.exists():
    raise FileNotFoundError(
        f"Data not found at {data_file}\n"
        f"  Project root: {project_root.resolve()}\n"
        f"  CWD: {cwd.resolve()}\n"
        f"  Put M5 CSVs in: project_root/data/m5/"
    )
print("Project root:", project_root.resolve())

In [ ]:
import importlib
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import src.config
import src.m5_data
importlib.reload(src.config)
importlib.reload(src.m5_data)  # pick up load_one_series / other changes without kernel restart
from src.config import Paths, TrainConfig
from src.features import build_supervised_features, add_calendar_features, feature_columns
from src.m5_data import build_merged_long, load_one_series
from src.registry import load_latest

In [ ]:
# Paths from project root (run cell above first so project_root exists)
paths = Paths.from_root(project_root)
cfg = TrainConfig()
os.makedirs(paths.reports_dir, exist_ok=True)

# Series to use (change if you want a different one)
SERIES_FORECAST = "FOODS_3_586_CA_3_validation"   # best performer
SERIES_DIAGNOSE = "FOODS_3_541_CA_3_validation"  # problem series

# Sanity check
assert Path(paths.sales_csv).exists(), f"sales_csv not found: {paths.sales_csv}"
print("Paths OK. You can run Section 1 or 2 below.")

## 1. Forecast plot (28-day actual vs predicted)

Loads the saved model for a series and plots the last 28 days: actual sales vs model predictions.

In [ ]:
def get_series_df(paths, series_id):
    """Load M5 for one series only (low memory; uses load_one_series)."""
    return load_one_series(paths, series_id)

In [ ]:
series_id = SERIES_FORECAST
print(f"Loading data for {series_id}...")
series_df = get_series_df(paths, series_id).sort_values("date").reset_index(drop=True)
series_df = add_calendar_features(series_df)
feat_df = build_supervised_features(series_df)

horizon = cfg.horizon_days
if len(feat_df) < horizon:
    raise ValueError(f"Series has only {len(feat_df)} rows; need at least {horizon}")

print(f"Loading latest model for {series_id}...")
model, metadata = load_latest(paths.models_dir, series_id)

tail = feat_df.tail(horizon).copy()
feats = feature_columns()
X = tail[feats].to_numpy()
actual = tail["sales"].to_numpy(dtype=float)
pred = model.predict(X).astype(float)
dates = tail["date"].values

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(dates, actual, label="Actual", color="tab:blue", marker="o", markersize=4)
ax.plot(dates, pred, label="Predicted", color="tab:orange", marker="s", markersize=4)
ax.set_xlabel("Date")
ax.set_ylabel("Sales")
ax.set_title(f"{series_id}\nLast {horizon}-day fit (trained model)")
ax.legend()
ax.grid(True, alpha=0.3)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

In [ ]:
# Optional: save figure
# out_path = os.path.join(paths.reports_dir, f"forecast_{series_id.replace('/', '_')}.png")
# plt.savefig(out_path, dpi=150)
# print(f"Saved: {out_path}")

## 2. Diagnose series (raw sales + stats + suggestion)

Loads raw sales for a series, plots the full series, and prints stats. For noisy or high-zero series, suggests naive fallback or extra features.

In [ ]:
series_id = SERIES_DIAGNOSE
print(f"Loading {series_id}...")
sdf = get_series_df(paths, series_id)
sdf["date"] = pd.to_datetime(sdf["date"])
sales = sdf["sales"].to_numpy(dtype=float)
dates = sdf["date"].values

n = len(sales)
mean_sales = float(np.mean(sales))
std_sales = float(np.std(sales))
pct_zero = 100 * (sales == 0).sum() / n
cv = (std_sales / mean_sales) if mean_sales > 0 else float("inf")

print("--- Stats ---")
print(f"  n_obs       {n}")
print(f"  mean        {mean_sales:.2f}")
print(f"  std         {std_sales:.2f}")
print(f"  cv          {cv:.2f}" + (" (high variability)" if cv > 1 else ""))
print(f"  % zeros     {pct_zero:.1f}%")
print(f"  min / max   {sales.min():.0f} / {sales.max():.0f}")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(dates, sales, color="tab:blue", alpha=0.8)
ax.set_xlabel("Date")
ax.set_ylabel("Sales")
ax.set_title(f"Raw sales: {series_id}")
ax.grid(True, alpha=0.3)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

In [ ]:
print("--- Suggestion ---")
if pct_zero > 30 or (mean_sales > 0 and cv > 1.2):
    print(
        "This series is noisy or has many zero days. Consider:\n"
        "  - Using naive (last value) as fallback when XGB CV MAPE is similar to naive.\n"
        "  - Adding features (e.g. more lags, event dummies) or a separate model for intermittent demand."
    )
else:
    print("Variability looks moderate. If CV MAPE is still high, try more lags or calendar/event features.")